# AE3 (S30): Transformación y Enriquecimiento de Datos

## 1. Necesidad, pregunta y alcance

La transformación prepara WA_Crash para analizar: **¿cómo se distribuye la severidad de los accidentes registrados entre 2019 y 2023 según el tipo de calzada, la presencia de intersección, la franja horaria y los tipos de vehículos involucrados?**

| Dimensión | Variables de origen | Resultado esperado |
|---|---|---|
| Periodo | `CRASH_DATE` | Accidentes fechados entre 2019 y 2023 |
| Severidad | `SEVERITY` | Distribución de accidentes por categoría |
| Tipo de calzada | `CWAY` | Comparación descriptiva por categoría vial |
| Intersección | `INTERSECTION_NO`, `INTERSECTION_DESC` | Estado registrado; faltante no equivale a ausencia |
| Franja horaria | `CRASH_TIME` | Hora HHMM para la posterior derivación de franjas |
| Vehículos/personas involucrados | columnas `TOTAL_*_INVOLVED` | Conteos por tipo para el enriquecimiento posterior |

**Limitaciones:** no hay datos de tráfico ni de exposición; por tanto, no se puede afirmar que un tipo de calzada sea más riesgoso. Una intersección no registrada no equivale a que no exista. Las comparaciones son descriptivas y se limitan a accidentes registrados.

## 2. Carga del dataset y copia inalterada

Se carga el split `train` de Hugging Face y se conserva una copia independiente para comparar el resultado. El dataset crudo no se exporta al repositorio.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

RAIZ_REPO = Path.cwd().resolve()
for candidata in (RAIZ_REPO, *RAIZ_REPO.parents):
    if (candidata / "AE3" / "src" / "clases_po").is_dir():
        RAIZ_REPO = candidata
        break
else:
    raise FileNotFoundError("No se encontró la raíz del repositorio con AE3/src/clases_po.")

SOURCE_DIR = RAIZ_REPO / "AE3" / "src"
RESULTADOS_DIR = RAIZ_REPO / "AE3" / "resultados"
sys.path.insert(0, str(SOURCE_DIR))

from clases_po.hf_loader import HFDatasetLoader
from clases_po.preprocesamiento import PreprocesadorCrash
from clases_po.reportes import Reportes

RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)
loader = HFDatasetLoader(dataset_name="kalley/WA_Crash", split="train")
df = loader.load()
df_original = df.copy(deep=True)
print(f"Dataset: {loader.info()}")
print(f"Columnas: {df_original.columns.tolist()}")

C:\Users\Personal\OneDrive\Escritorio\talleres de programacion\priyecto integrador\pp\PI_2026_IUD-1\.venv-ae2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset: {'dataset': 'kalley/WA_Crash', 'config': None, 'split': 'train', 'filas': 127102, 'columnas': 26}
Columnas: ['X', 'Y', 'ACC_ID', 'ROAD_NO', 'ROAD_NAME', 'COMMON_ROAD_NAME', 'CWAY', 'SLK', 'INTERSECTION_NO', 'INTERSECTION_DESC', 'LONGITUDE', 'LATITUDE', 'CRASH_DATE', 'CRASH_TIME', 'ACCIDENT_TYPE', 'SEVERITY', 'EVENT_NATURE', 'EVENT_TYPE', 'TOTAL_BIKE_INVOLVED', 'TOTAL_TRUCK_INVOLVED', 'TOTAL_HEAVY_TRUCK_INVOLVED', 'TOTAL_MOTOR_CYCLE_INVOLVED', 'TOTAL_OTHER_VEHICLES_INVOLVED', 'TOTAL_PEDESTRIANS_INVOLVED', 'OBJECTID', 'GlobalID']


## 3. Transformación y validación

El diagnóstico permite revisar los tipos y los valores faltantes de origen antes de convertir columnas. No se altera `df_original`.

In [2]:
preprocesador = PreprocesadorCrash()
diagnostico_inicial = preprocesador.diagnosticar(df_original)
display(diagnostico_inicial)

,columna,tipo_actual,no_nulos,nulos,porcentaje_nulos,ejemplo
0,X,float64,127102,0,0.00,115.907761
1,Y,float64,127102,0,0.00,-31.966608
2,ACC_ID,int64,127102,0,0.00,10972646
3,ROAD_NO,object,127102,0,0.00,1290190
4,ROAD_NAME,object,127102,0,0.00,Cookham Rd
5,COMMON_ROAD_NAME,object,127102,0,0.00,Cookham Rd
6,CWAY,object,127102,0,0.00,S
7,SLK,float64,127102,0,0.00,0.76
8,INTERSECTION_NO,float64,68123,58979,46.40,13935.0
9,INTERSECTION_DESC,object,68123,58979,46.40,Melville Mandurah Hwy & Cockburn Rd


**Interpretación del diagnóstico inicial:** la tabla registra, para cada variable, el tipo recibido, la cantidad y porcentaje de nulos y un valor de ejemplo. Los nulos de hora e intersección se conservarán; no se imputan ni se interpretan como ausencia de intersección.

In [3]:
df_limpio = preprocesador.procesar(df_original)
print(f"DataFrame limpio: {df_limpio.shape[0]} filas x {df_limpio.shape[1]} columnas")
validacion_correcta = preprocesador.validar(df_limpio)
assert validacion_correcta, "Falló al menos un control de validación; revisar los resultados."

DataFrame limpio: 127102 filas x 18 columnas
✔ Fechas completas dentro del periodo 2019-2023
✔ ACC_ID único y no nulo
✔ Tipos esperados de fechas, horas y conteos
✔ Horas no nulas válidas en formato HHMM
✔ Categorías de SEVERITY esperadas


La siguiente tabla muestra cuántos registros y qué porcentaje representan las categorías observadas en `SEVERITY` y `CWAY` dentro del DataFrame limpio.

In [4]:
filas_frecuencia = []
for dimension in ("SEVERITY", "CWAY"):
    conteos = df_limpio[dimension].value_counts(dropna=False)
    porcentajes = df_limpio[dimension].value_counts(dropna=False, normalize=True) * 100
    for categoria, cantidad in conteos.items():
        filas_frecuencia.append(
            {
                "dimension": dimension,
                "categoria": "Sin dato" if pd.isna(categoria) else categoria,
                "accidentes": int(cantidad),
                "porcentaje": round(float(porcentajes.loc[categoria]), 2),
            }
        )

tabla_frecuencias = pd.DataFrame(filas_frecuencia)
display(tabla_frecuencias)

,dimension,categoria,accidentes,porcentaje
0,SEVERITY,PDO Major,68868,54.18
1,SEVERITY,PDO Minor,34750,27.34
2,SEVERITY,Medical,16485,12.97
3,SEVERITY,Hospital,6264,4.93
4,SEVERITY,Fatal,735,0.58
5,CWAY,S,69954,55.04
6,CWAY,L,44758,35.21
7,CWAY,R,12390,9.75


**Interpretación de frecuencias:** sobre 127.102 accidentes, `SEVERITY` se distribuye en PDO Major: 68.868 (54,18 %), PDO Minor: 34.750 (27,34 %), Medical: 16.485 (12,97 %), Hospital: 6.264 (4,93 %) y Fatal: 735 (0,58 %). Para `CWAY`, S representa 69.954 (55,04 %), L 44.758 (35,21 %) y R 12.390 (9,75 %). Son proporciones descriptivas del conjunto limpio.

El log siguiente documenta cada regla y sus conteos, incluidos los controles sin problemas. Los filtros de fechas y duplicados justifican las reducciones de filas; las horas inválidas y conteos imposibles se convierten en nulos sin eliminar la fila.

In [5]:
tabla_decisiones = preprocesador.tabla_log()
display(tabla_decisiones)

,paso,regla,criterio/justificación,registros_afectados,filas_antes,filas_despues
0,fechas no interpretables,Excluir registros sin fecha interpretable,Sin fecha no es posible asignar el accidente a...,0,127102,127102
1,filtro de periodo,Conservar accidentes entre 2019 y 2023 inclusive,La pregunta de análisis limita el alcance a es...,0,127102,127102
2,duplicados completos,Conservar una copia de cada fila idéntica,Las repeticiones exactas no representan regist...,0,127102,127102
3,duplicados por ACC_ID,Conservar el primer registro de cada identific...,ACC_ID identifica el accidente; los registros ...,0,127102,127102
4,consistencia de intersección,Revisar el patrón de faltantes de INTERSECTION...,58979 registros tienen ambas variables nulas; ...,0,127102,127102
5,horas inválidas,Invalidar horas fuera de HHMM (0000-2359 y min...,Las horas inválidas se convierten en nulos; no...,0,127102,127102
6,normalización de texto,Eliminar espacios sobrantes al inicio y al final,Se recortan únicamente los valores que contení...,0,127102,127102
7,categorías de SEVERITY,Revisar variantes por mayúsculas/minúsculas o ...,No se encontraron categorías duplicadas por ma...,0,127102,127102
8,categorías de CWAY,Revisar variantes por mayúsculas/minúsculas o ...,No se encontraron categorías duplicadas por ma...,0,127102,127102
9,valores de SEVERITY,"Contrastar categorías con FATAL, HOSPITAL, MED...",No se eliminan ni sustituyen categorías descon...,0,127102,127102


**Interpretación del log:** `registros_afectados` expresa las filas o valores involucrados en la regla; `filas_antes` y `filas_despues` permiten distinguir los filtros de las revisiones que solo modifican valores o columnas. No se rellenan los faltantes de intersección ni de hora.

In [6]:
comparacion = preprocesador.comparar_antes_despues(df_original, df_limpio)
display(comparacion)

,columna,filas_antes,columnas_antes,tipo_antes,nulos_antes,filas_despues,columnas_despues,tipo_despues,nulos_despues
0,X,127102,26,float64,0,127102,18,descartada,NaN
1,Y,127102,26,float64,0,127102,18,descartada,NaN
2,ACC_ID,127102,26,int64,0,127102,18,int64,0.0
3,ROAD_NO,127102,26,object,0,127102,18,descartada,NaN
4,ROAD_NAME,127102,26,object,0,127102,18,descartada,NaN
5,COMMON_ROAD_NAME,127102,26,object,0,127102,18,descartada,NaN
6,CWAY,127102,26,object,0,127102,18,string,0.0
7,SLK,127102,26,float64,0,127102,18,descartada,NaN
8,INTERSECTION_NO,127102,26,float64,58979,127102,18,float64,58979.0
9,INTERSECTION_DESC,127102,26,object,58979,127102,18,string,58979.0


**Interpretación de la comparación:** las columnas descartadas aparecen como `descartada`; para las conservadas se muestran los tipos y nulos en ambos estados. El resultado contiene solo las variables pertinentes y mantiene los nombres originales en mayúsculas.

## SECCIÓN 4 y 5: ENRIQUECIMIENTO Y TABLAS (Persona B)

La Persona B agregará aquí las variables derivadas y las tablas de resultados. El punto de partida es `df_limpio`; la Persona B podrá reasignar `df_final` al DataFrame enriquecido. Este marcador no depende de módulos que aún no existen.

## 6. Resumen descriptivo del DataFrame final

Por ahora `df_final` corresponde al DataFrame limpio. Cuando termine el enriquecimiento, la Persona B debe asignar aquí el DataFrame enriquecido a esta única variable. Los reportes se regeneran en cada ejecución y se comprueba que coincidan con el DataFrame de esa ejecución.

In [7]:
# La Persona B puede reasignar esta única variable al DataFrame enriquecido.
df_final = df_limpio
reportes = Reportes()
ruta_resumen = RESULTADOS_DIR / "resumen_dataframe.txt"
ruta_metadata = RESULTADOS_DIR / "resumen_dataframe_metadata.txt"

texto_resumen = reportes.generar_reporte(
    df_final,
    dataset_id="kalley/WA_Crash",
    split="train",
    es_muestra=False,
)
estadisticas = df_final.describe(include="all").to_string()
texto_resumen += "\nEstadísticas descriptivas (todas las columnas):\n" + estadisticas + "\n"
ruta_resumen.write_text(texto_resumen, encoding="utf-8")

metadata_final = reportes.reporte_descriptivo(df_final)
reportes.guardar_reporte_metadata(metadata_final, ruta_metadata)

contenido_resumen = ruta_resumen.read_text(encoding="utf-8")
contenido_metadata = ruta_metadata.read_text(encoding="utf-8")
assert f"Filas: {len(df_final)}" in contenido_resumen
assert f"Columnas: {len(df_final.columns)}" in contenido_resumen
seccion_tipos = contenido_resumen.split("Columnas y tipos:\n", 1)[1].split("\n\nValores nulos por columna:", 1)[0]
seccion_nulos = contenido_resumen.split("Valores nulos por columna:\n", 1)[1].split("\n\nEstadísticas numéricas:", 1)[0]
assert all(f"- {columna}: {df_final[columna].dtype}" in seccion_tipos for columna in df_final.columns)
assert all(f"- {columna}: {int(df_final[columna].isna().sum())}" in seccion_nulos for columna in df_final.columns)
assert contenido_metadata == metadata_final.to_string(index=False) + "\n"
assert len(metadata_final) == len(df_final.columns)
assert ruta_resumen.stat().st_size > 0 and ruta_metadata.stat().st_size > 0
print(f"Reportes verificados: {ruta_resumen.relative_to(RAIZ_REPO)} y {ruta_metadata.relative_to(RAIZ_REPO)}")

Reportes verificados: AE3\resultados\resumen_dataframe.txt y AE3\resultados\resumen_dataframe_metadata.txt
